### LLM Inference-Serving Mechanics

`llm-architecture.ipynb` covers attention's training-time mechanics; this notebook covers the serving-time half of the same story: why autoregressive generation needs a KV-cache, why attention cost grows quadratically with context length (and what FlashAttention actually fixes about that), the "lost in the middle" long-context effect, and the main levers for controlling serving cost. `decoding-strategies.ipynb` covers the token-selection side (greedy/beam/sampling); this is about the compute/memory cost of getting to that point at all.

## KV-cache

#### 0. Core idea

Autoregressive generation produces one token at a time, and at every step, self-attention needs the Key and Value projections for every token generated so far. Without a cache, the naive approach recomputes K/V for the ENTIRE sequence-so-far at every single generation step, even though almost all of those tokens' K/V vectors are identical to what was just computed one step earlier, nothing about a past token's own K/V changes when a new token gets appended. The KV-cache just stores those K/V vectors once and reuses them, so each new step only computes K/V for the ONE new token.

Toy: a 3-token prompt, generating 5 new tokens one at a time.

In [ ]:
prompt_len = 3
n_generate = 5
seq_len_after_step = [prompt_len + i for i in range(1, n_generate + 1)]
print("sequence length after each generation step:", seq_len_after_step)

without_cache = sum(seq_len_after_step)          # recompute K/V for the whole sequence, every step
with_cache = prompt_len + n_generate             # encode the prompt once, then 1 new token's K/V per step

print("K/V projections computed WITHOUT a cache:", without_cache)
print("K/V projections computed WITH a cache:   ", with_cache)
print("savings ratio at this tiny scale:", round(without_cache / with_cache, 2))

#### 1. Reading the result

Even at this toy scale (8 total tokens) caching cuts redundant K/V computation by 3.75x, and the gap widens with sequence length, without a cache the redundant work grows roughly with the SQUARE of sequence length (summing `1+2+...+n`), while with a cache it only grows LINEARLY (one new token's worth of work per step). This is why every production LLM-serving stack (vLLM, TensorRT-LLM, HF `transformers` `use_cache=True`) keeps a per-request KV-cache by default, and why the KV-cache's own memory footprint (not just compute) becomes the dominant cost at long context lengths, covered below.

## Quadratic attention cost and FlashAttention

#### 0. Core idea

Self-attention computes a full `Q @ K^T` similarity matrix between every pair of tokens, an `n x n` matrix for sequence length `n`. That means both the compute AND the memory needed for the attention matrix itself scale with `n^2`, doubling the context length quadruples the attention cost.

In [ ]:
n1, n2 = 512, 1024  # doubling the context length
cost1, cost2 = n1**2, n2**2
print(f"attention matrix size at n={n1}: {cost1}")
print(f"attention matrix size at n={n2}: {cost2}")
print("ratio:", cost2 / cost1, " (doubling context length quadruples this cost)")

#### 1. What FlashAttention actually changes

FlashAttention does NOT change this `O(n^2)` compute complexity, the number of pairwise similarity scores that must be computed is unchanged. What it fixes is that the naive implementation MATERIALIZES the full `n x n` matrix in slow GPU memory (HBM) before the softmax and weighted sum, which is extremely memory-bandwidth-heavy since HBM reads/writes dominate the actual wall-clock cost, more than the raw FLOPs do. FlashAttention fuses the whole operation into blockwise chunks that stay in fast on-chip SRAM, computing the softmax incrementally so the full matrix is never written to HBM at all. Same asymptotic FLOP count, dramatically less memory traffic, which is what actually makes it faster and lets you fit longer sequences before running out of memory.

## "Lost in the middle"

#### 0. Core idea

An empirical finding about long-context LLM behavior (Liu et al., 2023, "Lost in the Middle"): given a long context stuffed with retrieval-relevant information, models are noticeably better at using information placed at the very START or very END of the context than information placed in the MIDDLE, even when the model's stated context window comfortably covers the whole input. This is a real limitation of how these models were trained and attend over long contexts, not something with a clean toy formula to hand-derive the way KV-cache math has, so no fabricated benchmark numbers here, just the practical takeaway.

#### 1. Practical implication

If you're building a RAG pipeline (`rag-pipeline.ipynb`) or stuffing several retrieved chunks into one prompt, don't assume the model will weigh a fact buried in chunk 4 of 8 as reliably as one at the very start or end. Common mitigations: put the highest-confidence / most-relevant retrieved chunk first or last rather than in position order, keep the total number of stuffed chunks small (fewer, better-ranked chunks over many marginal ones), or explicitly re-ask the model to re-state the key fact before answering, forcing it to actually attend to it rather than skim past it.

## Cost-tiering levers at serving

#### 0. Core idea

The KV-cache itself has a real memory cost, one that grows with context length and dictates how many concurrent requests a GPU can serve. Toy calculation, a ~7B-parameter-scale model config:

In [ ]:
n_layers, n_heads, head_dim, seq_len, bytes_per_elem = 32, 32, 128, 4096, 2  # fp16

# factor of 2 is for storing both K and V
kv_cache_bytes = 2 * n_layers * n_heads * head_dim * seq_len * bytes_per_elem
print("KV-cache size for one request at seq_len=4096:", round(kv_cache_bytes / 1e9, 2), "GB")

for sl in [2048, 4096, 8192]:
    size_gb = 2 * n_layers * n_heads * head_dim * sl * bytes_per_elem / 1e9
    print(f"  seq_len={sl}: {round(size_gb, 2)} GB per request")

#### 1. The levers this motivates

At ~2 GB per request just for the KV-cache at a 4096-token context, and that scaling LINEARLY with context length, the memory budget (not raw FLOPs) is usually what caps how many concurrent requests a serving fleet can handle, and the standard cost-tiering levers all attack this directly: **quantization** (storing the KV-cache and/or weights in int8/int4 instead of fp16 cuts this number roughly in half or quarter), **model routing** (send easy/cheap queries to a smaller distilled model, only route hard queries to the large model), **continuous batching** (pack many requests' KV-caches together so GPU memory and compute aren't idling between requests of different lengths), and **prefix caching** (if many requests share a common system prompt or retrieved context prefix, cache and reuse that prefix's K/V across requests instead of recomputing it every time).